# Per-Layer Sigmoid Gates for Multimodal Fusion

**Extension of DynMM (Xue & Marculescu, CVPR 2023)**

Instead of DynMM's single input-level gate that selects between pre-trained branches, we place an **independent sigmoid gate at each transformer layer for each modality**.  
This allows the network to learn patterns like:

> *"use text from layer 1, add audio at layer 3, skip video entirely"*

-- all on a **per-sample** basis, end-to-end from scratch.

### Gate mechanism

$$g_{m,l} = \sigma\!\left(W \cdot \text{mean\_pool}(h_{m,l})\right) \in (0,1)$$

Applied as a gated residual so gradients always flow:

$$h_{m,l+1} = g_{m,l}\; h_{m,l}^{\text{new}} + (1 - g_{m,l})\; h_{m,l}$$

$g \approx 1$ → modality fully updates at this layer &nbsp;&nbsp; $g \approx 0$ → modality passes through unchanged

### Dataset
CMU-MOSEI sentiment regression (visual=35, audio=74, text=300-d GloVe, seq_len≤50)

## 1 · Setup

> **Run this cell first, then restart the runtime** (`Runtime → Restart runtime`).  
> After restart, skip this cell and continue from the next one.

In [ ]:
# ⚠️  Run ONCE, then restart the runtime before continuing.
# Colab's default PyTorch (2.10+) has a broken _functorch module.
# Pin to 2.5.1 which is stable and fully compatible with MultiBench.
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import sys, os

# ── Colab: clone the repo if not already present ─────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.isdir('/content/MultiBench'):
        !git clone https://github.com/pliang279/MultiBench /content/MultiBench
    REPO_ROOT = '/content/MultiBench'
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repository root:', REPO_ROOT)

In [ ]:
# ── Checkpoint directory (persistent across Colab sessions via Drive) ─────────
if IN_COLAB:
    CHECKPOINT_DIR = '/content/drive/MyDrive/plg_checkpoints'
else:
    CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'log', 'mosei')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

try:
    import scipy
except ImportError:
    !pip install -q scipy

try:
    import sklearn
except ImportError:
    !pip install -q scikit-learn

import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import pearsonr

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 2 · Data

The CMU-MOSEI pickle file can be found at:  
**Google Drive folder** → `1A_hTmifi824gypelGobgl2M-5Rw9VWHv`

In [ ]:
# ── Mount Drive / set data path ───────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_FOLDER = '/content/drive/MyDrive/mosei'
    DATA_PATH = os.path.join(DRIVE_FOLDER, 'mosei_senti_data.pkl')

    if not os.path.exists(DATA_PATH):
        print('File not found in My Drive — downloading with gdown …')
        !pip install -q gdown
        import gdown
        gdown.download_folder(
            id='1A_hTmifi824gypelGobgl2M-5Rw9VWHv',
            output='/content/mosei_data',
            quiet=False,
        )
        DATA_PATH = '/content/mosei_data/mosei_senti_data.pkl'
else:
    DATA_PATH = os.path.join(REPO_ROOT, 'data', 'mosei_senti_data.pkl')

print('Data path:', DATA_PATH)
assert os.path.exists(DATA_PATH), f'File not found: {DATA_PATH}'

In [ ]:
# Patch out torchtext import (not needed when robust_test=False)
import importlib, types

fake_torchtext = types.ModuleType('torchtext')
fake_torchtext.vocab = types.ModuleType('torchtext.vocab')
sys.modules['torchtext'] = fake_torchtext
sys.modules['torchtext.vocab'] = fake_torchtext.vocab

In [ ]:
from datasets.affect.get_data import get_dataloader

BATCH_SIZE  = 32
MAX_SEQ_LEN = 50

traindata, validdata, testdata = get_dataloader(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    max_pad=True,
    max_seq_len=MAX_SEQ_LEN,
    robust_test=False,
    data_type='mosei',
    num_workers=0,
)

# Sanity check
v, a, t, y = next(iter(traindata))
print(f'vision  : {tuple(v.shape)}   (B, T, 35)')
print(f'audio   : {tuple(a.shape)}   (B, T, 74)')
print(f'text    : {tuple(t.shape)}   (B, T, 300)')
print(f'labels  : {tuple(y.shape)}')
print(f'train batches: {len(traindata)}  |  valid: {len(validdata)}  |  test: {len(testdata)}')

## 3 · Model

`PLGModel` wraps `PerLayerGatedFusion` (the encoder) with a linear regression head.

In [ ]:
from fusions.per_layer_gated import PLGModel

# Hyper-parameters
D_MODEL  = 64
N_HEADS  = 4
N_LAYERS = 4
DROPOUT  = 0.1
REG      = 0.01   # weight for gate sparsity regularisation

model = PLGModel(
    in_dims=[35, 74, 300],
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PLGModel  d_model={D_MODEL}  n_layers={N_LAYERS}  n_heads={N_HEADS}')
print(f'Trainable parameters: {n_params:,}')
print()
print(model)

In [ ]:
# ── Quick shape test (no GPU required) ────────────────────────────────────────
with torch.no_grad():
    dummy = [torch.randn(4, MAX_SEQ_LEN, d) for d in [35, 74, 300]]
    pred, reg = model(dummy)
    print(f'pred shape : {tuple(pred.shape)}   (B, 1)')
    print(f'reg_loss   : {reg.item():.4f}   (mean gate value — should be ~0.5 at init)')

    gates = model.encoder.forward(dummy)[1]
    print(f'gates shape: {tuple(gates.shape)}   (B, n_modalities, n_layers)')

## 4 · Training

Custom training loop (bypasses `Supervised_Learning.train()` which is
incompatible with newer PyTorch versions).

The total loss is:

$$\mathcal{L} = \underbrace{\text{MAE}(\hat{y}, y)}_{\text{task}} + \lambda \cdot \underbrace{\text{mean}(g_{m,l})}_{\text{gate sparsity}}$$

The sparsity term encourages the model to use fewer transformer layers per modality.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 10       # increase to 40 for full training
LR     = 1e-3
WD     = 1e-4
SAVE   = os.path.join(CHECKPOINT_DIR, 'plg_best.pt')

print(f'Training on {DEVICE} for {EPOCHS} epochs …')

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
criterion = torch.nn.L1Loss()
best_val  = float('inf')

model.to(DEVICE)

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    epoch_train_loss = 0.0
    for batch in traindata:
        *modalities, targets = batch
        modalities = [m.float().to(DEVICE) for m in modalities]
        targets = targets.float().to(DEVICE)

        optimizer.zero_grad()
        pred, reg_loss = model(modalities)
        loss = criterion(pred, targets) + REG * reg_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=8.0)
        optimizer.step()
        epoch_train_loss += loss.item()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for batch in validdata:
            *modalities, targets = batch
            modalities = [m.float().to(DEVICE) for m in modalities]
            targets = targets.float().to(DEVICE)
            pred, reg_loss = model(modalities)
            epoch_val_loss += (criterion(pred, targets) + REG * reg_loss).item()

    avg_train = epoch_train_loss / len(traindata)
    avg_val   = epoch_val_loss / len(validdata)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    marker = ''
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), SAVE)
        marker = '  ← best'

    print(f'Epoch {epoch+1:>2}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f}{marker}')

print(f'\nTraining complete. Best val loss: {best_val:.4f}, saved to {SAVE}')

In [ ]:
# ── Plot training curve ───────────────────────────────────────────────────────
if val_losses:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    epochs_range = range(1, len(val_losses) + 1)
    ax.plot(epochs_range, train_losses, marker='s', linewidth=2, markersize=5, label='Train')
    ax.plot(epochs_range, val_losses, marker='o', linewidth=2, markersize=5, label='Validation')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MAE + λ·reg)')
    ax.set_title('PLG Fusion – Training Curve (CMU-MOSEI)')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.show()

## 5 · Evaluate

In [ ]:
best_model = PLGModel(
    in_dims=[35, 74, 300],
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
)
best_model.load_state_dict(torch.load(SAVE, map_location=DEVICE))
best_model.to(DEVICE)
best_model.eval()

def evaluate_mae_corr(model, dataloader, device):
    """Compute MAE and Pearson correlation on a dataloader."""
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            v, a, t, y = batch
            inputs = [v.float().to(device), a.float().to(device), t.float().to(device)]
            pred, _ = model(inputs)
            all_preds.append(pred.squeeze(-1).cpu())
            all_labels.append(y.squeeze(-1).cpu())
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    mae  = float(np.abs(preds - labels).mean())
    corr, _ = pearsonr(preds, labels)
    acc  = float(np.mean((preds >= 0) == (labels >= 0)))
    return {'MAE': mae, 'Corr': corr, 'Acc (pos/neg)': acc}

val_results  = evaluate_mae_corr(best_model, validdata, DEVICE)
test_results = evaluate_mae_corr(best_model, testdata,  DEVICE)

print('┌─────────────────┬──────────┬──────────┬──────────────┐')
print('│ Split           │   MAE    │   Corr   │ Acc (pos/neg)│')
print('├─────────────────┼──────────┼──────────┼──────────────┤')
print(f"│ Validation      │  {val_results['MAE']:.4f}  │  {val_results['Corr']:.4f}  │    {val_results['Acc (pos/neg)']:.4f}    │")
print(f"│ Test            │  {test_results['MAE']:.4f}  │  {test_results['Corr']:.4f}  │    {test_results['Acc (pos/neg)']:.4f}    │")
print('└─────────────────┴──────────┴──────────┴──────────────┘')

## 5b · Comparison Against DynMM Baseline

We train a **Late Fusion (LF) baseline** using the identical architecture minus the gates: three independent Transformer encoders, each processing one modality, with the final hidden states concatenated and passed to a linear regression head.

This matches DynMM's **E2 branch** — the expensive "full multimodal" branch — and serves as the ceiling for straightforward late fusion.

Reference numbers from **DynMM Table 2** (Xue & Marculescu, CVPR 2023):

| Method         | MAE   | Acc (pos/neg) | MAdds   |
|----------------|-------|---------------|---------|
| Text-only  E1  | 0.620 | 78.35 %       | 124.7 M |
| Late Fusion E2 | 0.598 | 79.54 %       | 309.6 M |
| DynMM-a        | 0.622 | 79.07 %       | 165.5 M |
| DynMM-b        | 0.598 | 79.73 %       | 254.5 M |
| DynMM-c        | 0.598 | 79.75 %       | 295.8 M |

> **Note on MAdds for PLGModel**: the per-layer gates are *soft* (gated residual, not hard skip), so the forward pass runs all transformer layers for all modalities. Actual MAdds ≈ LF + a small gate overhead. The gain over LF is representational quality, not compute reduction.

## 6 · Visualise Gates

The gate heatmap below shows the **mean sigmoid gate value** across the test set for each **(modality, layer)** pair.

- **Warm colour (g → 1)**: the modality actively transforms its representation at this layer.  
- **Cool colour (g → 0)**: the modality passes through unchanged — effectively *skipped* at this depth.

This reveals the learned **fusion depth** per modality.

In [ ]:
# Collect gate values from the full test set
best_model.reset_weight()
best_model.eval()
with torch.no_grad():
    for batch in testdata:
        v, a, t, _ = batch
        inputs = [v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)]
        best_model(inputs)
best_model._tracking = False

mean_gates = best_model.get_mean_gates()  # [n_modalities, n_layers]
print('Mean gate values [modality × layer]:')
print(mean_gates.numpy().round(3))

In [ ]:
MODALITY_NAMES = ['Visual (35d)', 'Audio (74d)', 'Text (300d)']
n_mod, n_lay = mean_gates.shape

fig, ax = plt.subplots(figsize=(max(5, n_lay * 1.2), max(2.5, n_mod * 0.9)))

im = ax.imshow(mean_gates.numpy(), vmin=0.0, vmax=1.0,
               cmap='RdYlGn', aspect='auto')

for m in range(n_mod):
    for l in range(n_lay):
        val = mean_gates[m, l].item()
        colour = 'black' if 0.25 < val < 0.75 else 'white'
        ax.text(l, m, f'{val:.2f}', ha='center', va='center',
                fontsize=11, color=colour, fontweight='bold')

ax.set_xticks(range(n_lay))
ax.set_xticklabels([f'Layer {l+1}' for l in range(n_lay)])
ax.set_yticks(range(n_mod))
ax.set_yticklabels(MODALITY_NAMES[:n_mod])
ax.set_xlabel('Transformer layer depth →')
ax.set_title(
    f'Per-Layer Sigmoid Gates — CMU-MOSEI test set (mean over {len(testdata.dataset)} samples)\n'
    'Green = active (g≈1)   |   Red = skipped (g≈0)',
    pad=10,
)
cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cb.set_label('Mean gate value', rotation=270, labelpad=15)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'plg_gate_heatmap.png'), bbox_inches='tight')
plt.show()
print('Heatmap saved to', os.path.join(CHECKPOINT_DIR, 'plg_gate_heatmap.png'))

In [ ]:
# ── Per-sample gate distribution (box-plot) ───────────────────────────────────
best_model.reset_weight()
best_model.eval()
with torch.no_grad():
    for batch in testdata:
        v, a, t, _ = batch
        inputs = [v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)]
        best_model(inputs)
best_model._tracking = False

all_gates = torch.cat(best_model._gate_accum, dim=0)  # [N, M, L]
N = all_gates.shape[0]

fig, axes = plt.subplots(1, n_mod, figsize=(4 * n_mod, 4), sharey=True)
for m, (ax, name) in enumerate(zip(axes, MODALITY_NAMES[:n_mod])):
    data = [all_gates[:, m, l].numpy() for l in range(n_lay)]
    bp = ax.boxplot(data, patch_artist=True, medianprops={'color': 'black', 'lw': 2})
    cmap = plt.cm.RdYlGn
    for patch, l in zip(bp['boxes'], range(n_lay)):
        patch.set_facecolor(cmap(mean_gates[m, l].item()))
    ax.set_xticklabels([f'L{l+1}' for l in range(n_lay)])
    ax.set_xlabel('Layer')
    ax.set_title(name)
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
    ax.grid(True, axis='y', alpha=0.3)

axes[0].set_ylabel('Gate value g')
fig.suptitle(f'Gate distribution over {N} test samples', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'plg_gate_boxplots.png'), bbox_inches='tight')
plt.show()
print('Box-plot saved to', os.path.join(CHECKPOINT_DIR, 'plg_gate_boxplots.png'))

## 7 · Regularisation Sweep (Optional)

Higher `λ` encourages sparser (lower) gate values.  
Run this cell to compare gate sparsity vs. MAE across a range of `λ` values.

In [ ]:
REG_SWEEP    = [0.0, 0.001, 0.01, 0.1]
SWEEP_EPOCHS = 5

sweep_results = []

for lossw in REG_SWEEP:
    print(f'\n── λ = {lossw} ──────────────────────────────────────────')
    sweep_model = PLGModel(
        in_dims=[35, 74, 300],
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    sweep_opt = torch.optim.Adam(sweep_model.parameters(), lr=LR, weight_decay=WD)
    sweep_crit = torch.nn.L1Loss()
    save_path = os.path.join(CHECKPOINT_DIR, f'plg_sweep_{lossw}.pt')
    best_sweep_val = float('inf')

    for epoch in range(SWEEP_EPOCHS):
        sweep_model.train()
        for batch in traindata:
            *mods, tgt = batch
            mods = [m.float().to(DEVICE) for m in mods]
            tgt = tgt.float().to(DEVICE)
            sweep_opt.zero_grad()
            pred, reg = sweep_model(mods)
            loss = sweep_crit(pred, tgt) + lossw * reg
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sweep_model.parameters(), max_norm=8.0)
            sweep_opt.step()

        sweep_model.eval()
        vl = 0.0
        with torch.no_grad():
            for batch in validdata:
                *mods, tgt = batch
                mods = [m.float().to(DEVICE) for m in mods]
                tgt = tgt.float().to(DEVICE)
                pred, reg = sweep_model(mods)
                vl += (sweep_crit(pred, tgt) + lossw * reg).item()
        avg_vl = vl / len(validdata)
        if avg_vl < best_sweep_val:
            best_sweep_val = avg_vl
            torch.save(sweep_model.state_dict(), save_path)
        print(f'  Epoch {epoch+1}/{SWEEP_EPOCHS} val={avg_vl:.4f}')

    # Load best and evaluate
    sweep_model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    sweep_model.to(DEVICE)
    sweep_model.reset_weight()
    sweep_model.eval()
    with torch.no_grad():
        for batch in testdata:
            v, a, t, _ = batch
            sweep_model([v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)])
    sweep_model._tracking = False
    gates_mean = sweep_model.get_mean_gates().mean().item()
    res = evaluate_mae_corr(sweep_model, testdata, DEVICE)
    res['lossw'] = lossw
    res['mean_gate'] = gates_mean
    sweep_results.append(res)
    print(f'  MAE={res["MAE"]:.4f}  Corr={res["Corr"]:.4f}  mean_gate={gates_mean:.4f}')

# Plot
lws  = [r['lossw']     for r in sweep_results]
maes = [r['MAE']       for r in sweep_results]
mg   = [r['mean_gate'] for r in sweep_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(range(len(lws)), maes,  marker='s', color='steelblue')
ax1.set_xticks(range(len(lws))); ax1.set_xticklabels([str(l) for l in lws])
ax1.set_xlabel('λ (reg weight)'); ax1.set_ylabel('Test MAE'); ax1.set_title('MAE vs λ'); ax1.grid(alpha=0.3)
ax2.plot(range(len(lws)), mg, marker='D', color='darkorange')
ax2.set_xticks(range(len(lws))); ax2.set_xticklabels([str(l) for l in lws])
ax2.set_xlabel('λ (reg weight)'); ax2.set_ylabel('Mean gate value'); ax2.set_title('Gate sparsity vs λ'); ax2.grid(alpha=0.3)
plt.suptitle('Regularisation sweep — sparsity / accuracy trade-off')
plt.tight_layout()
plt.show()

---
## Summary

| Component | Where defined |
|---|---|
| `PerLayerGatedFusion` | `fusions/per_layer_gated.py` |
| `PLGModel` (moe_model wrapper) | `fusions/per_layer_gated.py` |
| Training script (CLI) | `examples/affect/affect_plg.py` |
| Data loader (unchanged) | `datasets/affect/get_data.py` |

### Differences from DynMM

| | DynMM | Per-Layer Gates (this work) |
|---|---|---|
| Gate granularity | 1 gate per sample (branch selection) | 1 gate per modality × layer × sample |
| Pre-training required | Yes (unimodal branches first) | No (end-to-end) |
| Interpretability | Which branch was used | Which modality was active at which depth |
| Regularisation | Penalise expensive branch weight | Penalise non-zero gate means |